# ARCHS4 reconstruction error — % coverage models, random subsamples (compute)

**Environment:** `clamp-analyses`

Evaluates how well CLAMPfull_hall reconstructs gene expression at each sample coverage level (1%, 5%, 10%, 25%, 50%, 75%, 100%), following Taroni 2018 (multi-plier).

For each coverage × seed combination:
1. Load the subsampled FBM and CLAMPfull_hall model
2. Row-normalize the expression data (z-score per gene)
3. Reconstruct with `GetReconstructedExprs(Z, B)`
4. Compute per-sample Spearman correlation with `GetReconstructionCorrelation`

Inputs: `output/01_model_building/04_archs4/06_bp_coverage_rshall/`

Outputs: `output/01_model_building/06_reconstruction_error/`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(ggplot2)
library(here)

source(here("config.R"))

## Helper functions (Taroni 2018 / multi-plier)

In [ ]:
# Compute per-sample Spearman correlations (input vs reconstructed)
# in column chunks to avoid materialising the full n_genes x n_samples matrices.
# X_fbm is already row (gene) z-scored at the source (01_archs4_preprocess.ipynb),
# so it is compared directly to the reconstruction with no further normalization.
get_chunked_cors <- function(X_fbm, Z, B,
                              chunk_size = 500, cor.method = "spearman") {
  n_samples <- ncol(X_fbm)
  cors      <- numeric(n_samples)
  for (start in seq(1, n_samples, by = chunk_size)) {
    end        <- min(start + chunk_size - 1L, n_samples)
    idx        <- start:end
    chunk      <- X_fbm[, idx, drop = FALSE]
    xhat_chunk <- Z %*% B[, idx, drop = FALSE]
    for (j in seq_along(idx)) {
      cors[idx[j]] <- cor(chunk[, j], xhat_chunk[, j], method = cor.method)
    }
    rm(chunk, xhat_chunk)
  }
  cors
}

## Parameters

In [ ]:
base_output_dir <- config$ARCHS4$DATASET_FOLDER
output_dir      <- here("output", "01_model_building", "06_reconstruction_error")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds     <- base_seed + 0:2
n_runs    <- length(seeds)

coverage_levels <- list(
  list(pct = "01",  subdir = "00_bp_coverage_hall_rs_01",  prefix = "rs1"),
  list(pct = "05",  subdir = "01_bp_coverage_hall_rs_05",  prefix = "rs5"),
  list(pct = "10",  subdir = "02_bp_coverage_hall_rs_10",  prefix = "rs10"),
  list(pct = "25",  subdir = "03_bp_coverage_hall_rs_25",  prefix = "rs25"),
  list(pct = "50",  subdir = "04_bp_coverage_hall_rs_50",  prefix = "rs50"),
  list(pct = "75",  subdir = "05_bp_coverage_hall_rs_75",  prefix = "rs75"),
  list(pct = "100", subdir = "06_bp_coverage_hall_rs_100", prefix = "rs100")
)

message("Output dir: ", output_dir)
message("Seeds: ", paste(seeds, collapse = ", "))

## Main loop: compute reconstruction correlation

In [ ]:
results_path <- file.path(output_dir, "reconstruction_error_random_results.rds")

if (file.exists(results_path)) {
  message("Results already exist, loading from: ", results_path)
  results_list <- readRDS(results_path)
  message("Loaded ", length(results_list), " entries")
} else {
  results_list <- list()

  for (cov in coverage_levels) {
    cov_pct    <- as.integer(cov$pct)
    cov_subdir <- cov$subdir
    cov_prefix <- cov$prefix

    for (run_idx in seq_len(n_runs)) {
      current_seed <- seeds[run_idx]
      seed_dir     <- file.path(base_output_dir, "06_bp_coverage_rshall",
                                cov_subdir,
                                paste0("hall_coverage_", cov_prefix, "_seed_", run_idx))

      message(strrep("=", 60))
      message("Coverage: ", cov_pct, "% | Seed: ", current_seed, " | ", seed_dir)

      if (!dir.exists(seed_dir)) {
        warning("Directory not found, skipping: ", seed_dir)
        next
      }

      tryCatch({
        clamp_full <- readRDS(file.path(seed_dir, "CLAMPfull_hall.rds"))
        Z_full     <- as.matrix(clamp_full$Z)
        B_full     <- as.matrix(clamp_full$B)
        n_genes    <- nrow(Z_full)
        n_samples  <- ncol(B_full)
        message("  Dimensions: ", n_genes, " genes x ", n_samples, " samples")

        bk_path <- file.path(seed_dir, "fbm_subsampled")
        if (!file.exists(paste0(bk_path, ".bk"))) {
          message("  fbm_subsampled not found, using fbm_filtered (full dataset)")
          bk_path <- file.path(base_output_dir, "01_archs4_preprocess", "fbm_filtered")
        }
        X_fbm <- FBM(
          nrow        = n_genes,
          ncol        = n_samples,
          type        = "double",
          backingfile = bk_path,
          create_bk   = FALSE
        )

        message("  CLAMPfull_hall reconstruction (chunked)...")
        cors_full <- get_chunked_cors(X_fbm, Z_full, B_full, chunk_size = 500L)
        message("  CLAMPfull median cor: ", round(median(cors_full, na.rm = TRUE), 4))

        results_list[[length(results_list) + 1]] <- list(
          coverage_pct = cov_pct,
          seed         = current_seed,
          run          = run_idx,
          model        = "CLAMPfull_hall",
          n_lvs        = ncol(Z_full),
          n_samples    = n_samples,
          median_cor   = median(cors_full, na.rm = TRUE),
          mean_cor     = mean(cors_full, na.rm = TRUE),
          cor_vector   = list(cors_full)
        )

        rm(clamp_full, X_fbm, Z_full, B_full, cors_full)
        gc()
      }, error = function(e) {
        message("  ERROR (skipping): ", conditionMessage(e))
      })
    }
  }

  message("\nAll runs complete.")
}

## Save results

In [ ]:
results_df <- dplyr::bind_rows(
  lapply(results_list, function(x) {
    data.frame(
      coverage_pct = x$coverage_pct,
      seed         = x$seed,
      run          = x$run,
      model        = x$model,
      n_lvs        = x$n_lvs,
      n_samples    = x$n_samples,
      median_cor   = x$median_cor,
      mean_cor     = x$mean_cor,
      stringsAsFactors = FALSE
    )
  })
)

saveRDS(results_list, file.path(output_dir, "reconstruction_error_random_results.rds"))
data.table::fwrite(results_df, file.path(output_dir, "reconstruction_error_random_summary.csv"))

message("Saved to ", output_dir)
print(results_df)